### Загрузка данных и модели кластеризации

In [7]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display

df = pd.read_csv('patient_segmentation_dataset.csv')
kmeans_model = joblib.load('best_model.kpl')

clust_features = ['Annual_Visits', 'Num_Chronic_Conditions', 'Avg_Billing_Amount']
df['Cluster'] = kmeans_model.predict(df[clust_features].values)

D:\Program\Anaconda\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator KMeans from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Функция для генерации визуализаций

In [8]:
def plot_dashboard(selected_cluster, age_range, gender):
    data = df[
        (df['Cluster'] == selected_cluster) & 
        (df['Age'].between(age_range[0], age_range[1])) & 
        (df['Gender'] == gender)
    ]
    
    if data.empty:
        print("Нет данных для отображения")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    plt.subplots_adjust(wspace=0.3, hspace=0.3)

    axes[0, 0].scatter(data['Annual_Visits'], data['Avg_Billing_Amount'], alpha=0.5)
    axes[0, 0].set_title('Траты vs Визиты')
    
    axes[0, 1].hist(data['BMI'], bins=20, color='skyblue', edgecolor='black')
    axes[0, 1].set_title('Распределение BMI')
    
    care_data = data['Preventive_Care_Flag'].value_counts()
    axes[0, 2].pie(care_data, labels=care_data.index, autopct='%1.1f%%')
    axes[0, 2].set_title('Доля профилактики')
    
    axes[1, 0].boxplot(data['Age'])
    axes[1, 0].set_title('Разброс возраста')
    
    chronic_data = data['Num_Chronic_Conditions'].value_counts().sort_index()
    axes[1, 1].bar(chronic_data.index.astype(str), chronic_data.values, color='salmon')
    axes[1, 1].set_title('Хронические заболевания')

    fig.delaxes(axes[1, 2])
    plt.show()

### Создание интерфейса и запуск интерактивности

In [11]:
cluster_selector = widgets.Dropdown(
    options=sorted(df['Cluster'].unique()),
    description='Кластер:'
)

age_slider = widgets.IntRangeSlider(
    min=int(df['Age'].min()),
    max=int(df['Age'].max()),
    value=[20, 70],
    description='Возраст:'
)

gender_selector = widgets.SelectionSlider(
    options=['Male', 'Female'],
    description='Пол:'
)

interact(plot_dashboard, selected_cluster=cluster_selector, age_range=age_slider, gender=gender_selector)

interactive(children=(Dropdown(description='Кластер:', options=(np.int32(0), np.int32(2)), value=np.int32(0)),…

<function __main__.plot_dashboard(selected_cluster, age_range, gender)>